In [ ]:
# RQ2: how does the verifiable/baseline overhead vary with database size and record size? (spec 2, H2)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import frame, plots
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
t = plots.build(df, status)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


def save_table(name, table):
    (FIG / f"{name}.md").write_text(plots._markdown(table, plots.TABLE_TITLES[name], False), encoding="utf-8")


plots._style()
print(f"{len(df)} measurement rows from {', '.join(r.name for r in RUNS)}; {len(status)} (cell, phase) rows not ok")

In [ ]:
# T4: overhead = median(verifiable) / median(baseline) per cell, with paired bootstrap 95 % CIs (10 000 resamples),
# for the three price pairs and the S0-go/S0-cpp implementation ablation. Cells padded above 1.1 are flagged
# (decision 19).
t4 = plots.t4_overheads(t)
save_table("T4_overheads", t4)
METRICS = list(plots.F2_METRICS)
view = t.overheads[t.overheads.metric.isin(METRICS)]
for (pair, metric), g in view.groupby(["pair", "metric"]):
    tab = g.pivot_table(index="db_bytes", columns="record_bytes", values="ratio")
    tab.index = [fmt_size(int(i)) for i in tab.index]
    tab.columns = [fmt_size(int(c)) for c in tab.columns]
    print(f"{pair:14s} {metric}\n{tab.round(3).to_string()}\n")

In [ ]:
# H2: the overhead is largest for small records and small databases, and shrinks with database size. Per pair and
# metric: is the largest ratio at the smallest (database, record) cell measured, and does the ratio fall from the
# smallest to the largest database at each record size?
verdicts = []
for (pair, metric), g in view.groupby(["pair", "metric"]):
    g = g.dropna(subset=["ratio"])
    if g.empty:
        continue
    top = g.loc[g.ratio.idxmax()]
    smallest = (g.db_bytes.min(), g.record_bytes.min())
    falls = [g[g.record_bytes == r].sort_values("db_bytes").ratio.iloc[-1] < g[g.record_bytes == r].sort_values("db_bytes").ratio.iloc[0]
             for r in sorted(g.record_bytes.unique()) if (g.record_bytes == r).sum() > 1]
    verdicts.append({"pair": pair, "metric": metric, "max_ratio": round(top.ratio, 3),
                     "max_at": f"{fmt_size(int(top.db_bytes))} x {fmt_size(int(top.record_bytes))}",
                     "max_at_smallest_cell": (top.db_bytes, top.record_bytes) == smallest,
                     "falls_with_db_at": f"{sum(falls)} of {len(falls)} record sizes"})
print(pd.DataFrame(verdicts).to_string(index=False))

In [ ]:
# F2 (overhead heatmaps), F3 (absolute cost against size, S4 extrapolated past 1 MiB) and F6 (where the
# verification cost goes, at Q = 100).
for name in ("F2_overhead_heatmaps", "F3_absolute_cost", "F6_breakdown"):
    save(name, plots.FIGURE_BUILDERS[name](t))
print(sorted(p.name for p in FIG.glob("F[236]_*")))
print("S4 fit exponents:", t.fits.set_index("metric").exponent.round(3).to_dict())